# 04_208 · Auditoría común de etiquetas finas y transversales

Este cuaderno **no entrena** ni altera modelos. Evalúa todos los resultados disponibles sobre el mismo `test` 4:1 y deja explícito qué papel cumplen las anotaciones auxiliares.

- En `04_205`, las etiquetas finas y los flags se usan como **objetivos auxiliares predichos** durante el fine-tuning de Qwen.
- En `04_206`, sus logits predichos se reutilizan como características latentes; nunca se inyectan etiquetas gold en inferencia.
- En los modelos clásicos y Transformers restantes, el entrenamiento es `coarse-only`; las etiquetas finas y transversales se usan aquí para auditar subgrupos, errores y priorización de revisión humana.

Usar las etiquetas auxiliares gold como predictores produciría fuga de información, porque no estarán disponibles para chunks nuevos en producción.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name.lower() == 'cuadernos':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from IPython.display import display, Markdown, Image
from scripts_auxiliares import analizar_auxiliares_modelos_4 as audit4

## 1. Inventario antes de ejecutar

La celda siguiente permite saber qué experimentos ya terminaron. Los resultados ausentes quedan registrados; no se inventan ni se sustituyen por ejecuciones parciales.

In [ ]:
available, missing = audit4.collect_models()
display({
    'modelos_disponibles': [m['label'] for m in available],
    'regimen_de_supervision': {m['label']: m['regime'] for m in available},
    'resultados_pendientes': missing,
})

## 2. Ejecutar la auditoría reproducible

Para cada etiqueta fina se calcula recall de su categoría gruesa (o especificidad para variantes de `SEGURO`). Para cada flag transversal se mide qué proporción queda capturada al revisar el 10% y 20% de chunks más cercanos a los umbrales de decisión. Esta es una auditoría descriptiva común, no una prueba causal de que la supervisión auxiliar sea responsable de una diferencia.

In [ ]:
result = audit4.run_analysis(review_fractions=(0.10, 0.20))
display(result)

In [ ]:
import pandas as pd

fine = pd.read_csv(audit4.OUTPUT_DIR / 'desempeno_por_etiqueta_fina.csv')
flags = pd.read_csv(audit4.OUTPUT_DIR / 'captura_flags_por_incertidumbre.csv')
display(fine.sort_values(['fine_label', 'value'], ascending=[True, False]))
display(flags.sort_values(['review_fraction', 'flag', 'flag_capture'], ascending=[True, True, False]))

In [ ]:
for figure in (audit4.FIGURE_DIR / 'desempeno_fino.png', audit4.FIGURE_DIR / 'captura_flags.png'):
    if figure.exists():
        display(Image(filename=str(figure)))
display(Markdown(f'Informe reproducible: `{audit4.REPORT_PATH.relative_to(ROOT)}`'))

## Interpretación

La comparación principal entre modelos sigue siendo por las cuatro etiquetas gruesas y `SEGURO`. Esta auditoría responde dos preguntas complementarias: (1) si el desempeño se mantiene en cada fenómeno fino y (2) si los flags transversales se concentran entre los casos que el modelo enviaría a revisión. Para afirmar que la supervisión auxiliar mejora causalmente el modelo se requiere comparar una ablación Qwen idéntica con y sin las pérdidas auxiliares.